<a href="https://colab.research.google.com/github/nisreenabusalah-cmyk/assignments-session-2-/blob/main/Assignment_Clinical_Intake_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session2/students/Assignment_Clinical_Intake_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 Assignment: The Complete Clinical Intake Pipeline
## CCI Prompt Engineering & Clinical AI Development

**Due:** Before Session 3  
**Estimated effort:** 3-5 hours  
**Submission:** Push to your `cci-course-notebooks` GitHub repo under `assignments/session-2/`

### Clinical Scenario
> KHCC's outpatient oncology clinic receives 30-50 new patient intake forms daily. Each form contains demographics, initial lab results, current medications, and a brief clinical note. Currently, a data entry clerk manually reviews each form, flags abnormal labs, checks for drug interactions, and prepares a summary report for the oncologist. This takes 2-3 hours per day. You have been asked to build a Python notebook that automates this workflow.

### Grading
| Part | Weight | Description |
|------|--------|-------------|
| Part 1 | 30% | Build the Intake Data Model |
| Part 2 | 30% | Merge, Analyze, and Export |
| Part 3 | 20% | Analysis & Critical Thinking |
| Part 4 | 20% | Stretch Challenge |

### Anti-Shortcut Rules
- You MUST use 15+ unique patients with varied, realistic data
- Your Part 3 analysis must reference YOUR specific implementation
- All three CSV files must be produced by running the notebook top-to-bottom
- Push to GitHub with at least 3 separate commits with descriptive messages

---
## Section 1: Package Installations

In [ ]:
# === SECTION 1: PACKAGE INSTALLATIONS ===
# TODO: Install any packages you need (pandas is pre-installed in Colab)


---
## Section 2: Imports

In [ ]:
# === SECTION 2: IMPORTS ===
# TODO: Import the modules you will need
# At minimum: pandas, csv, datetime


---
## Section 3: Configuration

In [ ]:
# === SECTION 3: CONFIGURATION ===
# TODO: Define clinical thresholds
HGB_LOW = 10.0
HGB_CRITICAL = 7.0
WBC_HIGH = 11.0
WBC_LOW = 4.0
PLATELETS_LOW = 150
CREATININE_HIGH = 1.2

# TODO: Define output file names for the 3 CSVs
ALL_PATIENTS_FILE = 'all_patients.csv'
HIGH_RISK_PATIENTS_FILE = 'high_risk_patients.csv'
DRUG_INTERACTION_ALERTS_FILE = 'drug_interaction_alerts.csv'


---
## Section 4: Prompts

In [ ]:
# === SECTION 4: PROMPTS ===
# TODO: Define any prompt templates for future LLM integration
# Example: A prompt to summarize the daily intake report


---
## Section 5: Functions

### Part 1 -- PatientIntake Class (30%)

Define a `PatientIntake` class with:
- **Attributes:** name, mrn, age, gender, diagnosis, hemoglobin, wbc, platelets, creatinine, medications (list of strings), clinical_note (string)
- **Methods:**
  - `get_lab_alerts()` -- returns a list of alert strings for abnormal lab values
  - `is_high_risk()` -- returns True if multiple criteria are met (e.g., low HGB + abnormal WBC + on chemo)
  - `summary()` -- returns a formatted string with all patient info
  - `to_dict()` -- returns a dictionary for CSV export

In [ ]:
# === SECTION 5: FUNCTIONS ===

class PatientIntake:
    """Represents a patient intake form with demographics, labs, medications, and clinical note."""

    def __init__(self, name, mrn, age, gender, diagnosis, hemoglobin, wbc,
                 platelets, creatinine, medications=None, clinical_note=""):
        self.name = name
        self.mrn = mrn
        self.age = age
        self.gender = gender
        self.diagnosis = diagnosis
        self.hemoglobin = hemoglobin
        self.wbc = wbc
        self.platelets = platelets
        self.creatinine = creatinine
        self.medications = medications if medications is not None else []
        self.clinical_note = clinical_note

    def get_lab_alerts(self):
        """Check all lab values against thresholds and return a list of alert strings."""
        alerts = []
        if self.hemoglobin < HGB_CRITICAL:
            alerts.append(f"CRITICAL Hemoglobin: {self.hemoglobin} g/dL")
        elif self.hemoglobin < HGB_LOW:
            alerts.append(f"LOW Hemoglobin: {self.hemoglobin} g/dL")

        if self.wbc > WBC_HIGH:
            alerts.append(f"HIGH WBC: {self.wbc} K/uL")
        elif self.wbc < WBC_LOW:
            alerts.append(f"LOW WBC: {self.wbc} K/uL")

        if self.platelets < PLATELETS_LOW:
            alerts.append(f"LOW Platelets: {self.platelets} K/uL")

        if self.creatinine > CREATININE_HIGH:
            alerts.append(f"HIGH Creatinine: {self.creatinine} mg/dL")
        return alerts

    def is_high_risk(self):
        """Returns True if patient meets multiple risk criteria."""
        alerts = self.get_lab_alerts()
        # High risk criteria examples:
        # 1. Critical hemoglobin alone
        # 2. 2 or more abnormal labs
        # 3. Low WBC (neutropenia) and on chemotherapy (e.g., Cisplatin)

        if f"CRITICAL Hemoglobin: {self.hemoglobin} g/dL" in alerts:
            return True
        if len(alerts) >= 2:
            return True
        if f"LOW WBC: {self.wbc} K/uL" in alerts and any(med in self.medications for med in ["Cisplatin", "Carboplatin", "Paclitaxel", "Doxorubicin"]):
            return True
        return False

    def summary(self):
        """Returns a formatted summary string."""
        meds = ', '.join(self.medications) if self.medications else 'None'
        summary_str = f"""
Patient: {self.name} (MRN: {self.mrn})
Age: {self.age}, Gender: {self.gender}
Diagnosis: {self.diagnosis}
Hemoglobin: {self.hemoglobin} g/dL, WBC: {self.wbc} K/uL, Platelets: {self.platelets} K/uL, Creatinine: {self.creatinine} mg/dL
Medications: {meds}
Clinical Note: {self.clinical_note}
"""
        return summary_str

    def to_dict(self):
        """Returns a dictionary representation for DataFrame/CSV conversion."""
        return {
            'name': self.name,
            'mrn': self.mrn,
            'age': self.age,
            'gender': self.gender,
            'diagnosis': self.diagnosis,
            'hemoglobin': self.hemoglobin,
            'wbc': self.wbc,
            'platelets': self.platelets,
            'creatinine': self.creatinine,
            'medications': ', '.join(self.medications),
            'clinical_note': self.clinical_note,
            'is_high_risk': self.is_high_risk(),
            'alert_count': len(self.get_lab_alerts())
        }


---
## Section 6: Main Code

### Create 15+ Patient Intake Records
Use varied, realistic oncology data. Include:
- Some patients with all normal values
- Some with critical values
- Some with multiple abnormalities
- Different diagnoses (at least 4 different cancer types)
- Both male and female patients
- Different medications

In [ ]:
# === SECTION 6: MAIN CODE ===

# TODO: Create at least 15 PatientIntake objects
# Use varied, realistic KHCC oncology data
patients = [
    PatientIntake("Ahmad", "P-1001", 58, "M", "Lung Cancer",
                   hemoglobin=9.1, wbc=6.2, platelets=180, creatinine=1.1,
                   medications=["Cisplatin", "Pemetrexed"],
                   clinical_note="New diagnosis, stage IIIA NSCLC"),
    PatientIntake("Fatima", "P-1002", 65, "F", "Breast Cancer",
                   hemoglobin=12.5, wbc=7.8, platelets=250, creatinine=0.9,
                   medications=["Tamoxifen"],
                   clinical_note="ER/PR+, HER2- breast cancer, post-surgery"),
    PatientIntake("Omar", "P-1003", 72, "M", "Prostate Cancer",
                   hemoglobin=11.8, wbc=5.5, platelets=200, creatinine=1.3,
                   medications=["Enzalutamide"],
                   clinical_note="Metastatic prostate cancer, elevated creatinine"),
    PatientIntake("Layla", "P-1004", 45, "F", "Lymphoma",
                   hemoglobin=8.5, wbc=3.8, platelets=120, creatinine=0.8,
                   medications=["Rituximab", "Cyclophosphamide"],
                   clinical_note="Newly diagnosed Non-Hodgkin Lymphoma, pancytopenia"),
    PatientIntake("Yousef", "P-1005", 60, "M", "Colorectal Cancer",
                   hemoglobin=10.5, wbc=9.0, platelets=300, creatinine=1.0,
                   medications=["Fluorouracil", "Leucovorin"],
                   clinical_note="Stage III CRC, receiving adjuvant chemo"),
    PatientIntake("Sara", "P-1006", 53, "F", "Ovarian Cancer",
                   hemoglobin=7.5, wbc=2.5, platelets=90, creatinine=1.0,
                   medications=["Carboplatin", "Paclitaxel"],
                   clinical_note="Recurrent ovarian cancer, severe cytopenias"),
    PatientIntake("Khalid", "P-1007", 68, "M", "Melanoma",
                   hemoglobin=13.0, wbc=6.0, platelets=220, creatinine=1.1,
                   medications=["Pembrolizumab"],
                   clinical_note="Metastatic melanoma, immunotherapy start"),
    PatientIntake("Nour", "P-1008", 35, "F", "Thyroid Cancer",
                   hemoglobin=14.0, wbc=8.5, platelets=280, creatinine=0.7,
                   medications=["Levothyroxine"],
                   clinical_note="Post-thyroidectomy, well controlled"),
    PatientIntake("Ali", "P-1009", 70, "M", "Pancreatic Cancer",
                   hemoglobin=9.8, wbc=12.5, platelets=170, creatinine=1.5,
                   medications=["Gemcitabine"],
                   clinical_note="Advanced pancreatic cancer, elevated WBC and creatinine"),
    PatientIntake("Hana", "P-1010", 49, "F", "Breast Cancer",
                   hemoglobin=11.2, wbc=5.0, platelets=190, creatinine=0.9,
                   medications=["Trastuzumab"],
                   clinical_note="HER2+ breast cancer, maintenance therapy"),
    PatientIntake("Jamal", "P-1011", 62, "M", "Kidney Cancer",
                   hemoglobin=15.0, wbc=7.0, platelets=350, creatinine=1.0,
                   medications=["Sunitinib"],
                   clinical_note="Renal cell carcinoma, stable"),
    PatientIntake("Rania", "P-1012", 55, "F", "Multiple Myeloma",
                   hemoglobin=8.0, wbc=3.0, platelets=100, creatinine=1.8,
                   medications=["Dexamethasone", "Lenalidomide"],
                   clinical_note="Relapsed multiple myeloma, kidney involvement"),
    PatientIntake("Samir", "P-1013", 78, "M", "Lung Cancer",
                   hemoglobin=10.0, wbc=4.5, platelets=160, creatinine=1.2,
                   medications=["Osimertinib"],
                   clinical_note="EGFR+ NSCLC, stable on targeted therapy"),
    PatientIntake("Yasmin", "P-1014", 40, "F", "Leukemia",
                   hemoglobin=6.5, wbc=2.0, platelets=50, creatinine=0.8,
                   medications=["Imatinib"],
                   clinical_note="CML in blast crisis, severe cytopenias"),
    PatientIntake("Ziad", "P-1015", 63, "M", "Head & Neck Cancer",
                   hemoglobin=12.0, wbc=8.0, platelets=230, creatinine=1.1,
                   medications=["Cetuximab"],
                   clinical_note="Locally advanced HNC, post-radiation")
]

print(f"Total patients: {len(patients)}")


Total patients: 15


In [ ]:
# Print summaries and alerts for all patients
for p in patients:
    print(p.summary())
    alerts = p.get_lab_alerts()
    if alerts:
        print(f"  Alerts: {', '.join(alerts)}")
    if p.is_high_risk():
        print("  *** HIGH RISK ***")
    print()


Patient: Ahmad (MRN: P-1001)
Age: 58, Gender: M
Diagnosis: Lung Cancer
Hemoglobin: 9.1 g/dL, WBC: 6.2 K/uL, Platelets: 180 K/uL, Creatinine: 1.1 mg/dL
Medications: Cisplatin, Pemetrexed
Clinical Note: New diagnosis, stage IIIA NSCLC

  Alerts: LOW Hemoglobin: 9.1 g/dL


Patient: Fatima (MRN: P-1002)
Age: 65, Gender: F
Diagnosis: Breast Cancer
Hemoglobin: 12.5 g/dL, WBC: 7.8 K/uL, Platelets: 250 K/uL, Creatinine: 0.9 mg/dL
Medications: Tamoxifen
Clinical Note: ER/PR+, HER2- breast cancer, post-surgery



Patient: Omar (MRN: P-1003)
Age: 72, Gender: M
Diagnosis: Prostate Cancer
Hemoglobin: 11.8 g/dL, WBC: 5.5 K/uL, Platelets: 200 K/uL, Creatinine: 1.3 mg/dL
Medications: Enzalutamide
Clinical Note: Metastatic prostate cancer, elevated creatinine

  Alerts: HIGH Creatinine: 1.3 mg/dL


Patient: Layla (MRN: P-1004)
Age: 45, Gender: F
Diagnosis: Lymphoma
Hemoglobin: 8.5 g/dL, WBC: 3.8 K/uL, Platelets: 120 K/uL, Creatinine: 0.8 mg/dL
Medications: Rituximab, Cyclophosphamide
Clinical Note: Ne

---
### Part 2 -- Merge, Analyze, and Export (30%)

#### Step A: Convert patients to DataFrame

In [ ]:
# Step A: Convert to DataFrame
# TODO: Use list comprehension to call to_dict() on each patient
# Then create a DataFrame from the list of dicts
import pandas as pd

df = pd.DataFrame([p.to_dict() for p in patients])
print(df.head())
print(f"\nShape: {df.shape}")


     name     mrn  age gender          diagnosis  hemoglobin  wbc  platelets  \
0   Ahmad  P-1001   58      M        Lung Cancer         9.1  6.2        180   
1  Fatima  P-1002   65      F      Breast Cancer        12.5  7.8        250   
2    Omar  P-1003   72      M    Prostate Cancer        11.8  5.5        200   
3   Layla  P-1004   45      F           Lymphoma         8.5  3.8        120   
4  Yousef  P-1005   60      M  Colorectal Cancer        10.5  9.0        300   

   creatinine                  medications  \
0         1.1        Cisplatin, Pemetrexed   
1         0.9                    Tamoxifen   
2         1.3                 Enzalutamide   
3         0.8  Rituximab, Cyclophosphamide   
4         1.0     Fluorouracil, Leucovorin   

                                       clinical_note  is_high_risk  \
0                    New diagnosis, stage IIIA NSCLC         False   
1          ER/PR+, HER2- breast cancer, post-surgery         False   
2    Metastatic prostate cancer,

#### Step B: Create Drug Interaction Table

In [ ]:
# Step B: Drug interaction table
# TODO: Create a DataFrame with at least 10 drug pairs that interact
# Columns: drug1, drug2, interaction_severity, description
# Example interactions:
# Cisplatin + Metformin -> Nephrotoxicity risk
# Warfarin + Ibuprofen -> Bleeding risk

interactions = pd.DataFrame([
    {'drug1': 'Cisplatin', 'drug2': 'Metformin', 'interaction_severity': 'High', 'description': 'Increased risk of nephrotoxicity'},
    {'drug1': 'Warfarin', 'drug2': 'Ibuprofen', 'interaction_severity': 'High', 'description': 'Increased risk of bleeding'},
    {'drug1': 'Methotrexate', 'drug2': 'NSAIDs', 'interaction_severity': 'High', 'description': 'Increased methotrexate toxicity'},
    {'drug1': 'Tamoxifen', 'drug2': 'Paroxetine', 'interaction_severity': 'Moderate', 'description': 'Reduced tamoxifen efficacy'},
    {'drug1': 'Doxorubicin', 'drug2': 'Trastuzumab', 'interaction_severity': 'Moderate', 'description': 'Increased cardiotoxicity risk'},
    {'drug1': 'Cyclophosphamide', 'drug2': 'Allopurinol', 'interaction_severity': 'High', 'description': 'Increased bone marrow suppression'},
    {'drug1': 'Paclitaxel', 'drug2': 'Cisplatin', 'interaction_severity': 'Moderate', 'description': 'Altered drug clearance, increased neurotoxicity'},
    {'drug1': 'Imatinib', 'drug2': 'Warfarin', 'interaction_severity': 'High', 'description': 'Increased bleeding risk (CYP3A4 inhibition)'},
    {'drug1': 'Pembrolizumab', 'drug2': 'Corticosteroids', 'interaction_severity': 'Moderate', 'description': 'May reduce efficacy of immunotherapy if used systemically'},
    {'drug1': 'Lenalidomide', 'drug2': 'Digoxin', 'interaction_severity': 'Low', 'description': 'Increased digoxin levels'}
])

print(interactions)


              drug1            drug2 interaction_severity  \
0         Cisplatin        Metformin                 High   
1          Warfarin        Ibuprofen                 High   
2      Methotrexate           NSAIDs                 High   
3         Tamoxifen       Paroxetine             Moderate   
4       Doxorubicin      Trastuzumab             Moderate   
5  Cyclophosphamide      Allopurinol                 High   
6        Paclitaxel        Cisplatin             Moderate   
7          Imatinib         Warfarin                 High   
8     Pembrolizumab  Corticosteroids             Moderate   
9      Lenalidomide          Digoxin                  Low   

                                         description  
0                   Increased risk of nephrotoxicity  
1                         Increased risk of bleeding  
2                    Increased methotrexate toxicity  
3                         Reduced tamoxifen efficacy  
4                      Increased cardiotoxicity risk 

#### Step C: Check for Drug Interactions

In [ ]:
# Step C: Match patient medications against interaction table
# TODO: For each patient, check if any of their medications appear in
# the drug interaction table (as drug1 or drug2)
# Build a list of interaction alerts

drug_interaction_alerts = []

for patient in patients:
    patient_meds = set(patient.medications)
    for index, row in interactions.iterrows():
        drug1 = row['drug1']
        drug2 = row['drug2']

        # Check if both interacting drugs are in the patient's medications
        if drug1 in patient_meds and drug2 in patient_meds:
            alert = {
                'mrn': patient.mrn,
                'patient_name': patient.name,
                'drug1': drug1,
                'drug2': drug2,
                'interaction_severity': row['interaction_severity'],
                'description': row['description']
            }
            drug_interaction_alerts.append(alert)

drug_interactions_df = pd.DataFrame(drug_interaction_alerts)
print("Drug Interaction Alerts:")
print(drug_interactions_df.head())
print(f"\nTotal drug interaction alerts: {len(drug_interaction_alerts)}")

Drug Interaction Alerts:
Empty DataFrame
Columns: []
Index: []

Total drug interaction alerts: 0


#### Step D: Group Analysis

In [ ]:
# Step D: Group by diagnosis and calculate summary statistics
# TODO: Use df.groupby('diagnosis') to calculate mean, min, max
# for hemoglobin, wbc, platelets, creatinine

print("Summary statistics by diagnosis:")
# TODO: Print the groupby results

summary_stats = df.groupby('diagnosis')[['hemoglobin', 'wbc', 'platelets', 'creatinine']].agg(['mean', 'min', 'max'])
print(summary_stats)


Summary statistics by diagnosis:
                   hemoglobin                wbc             platelets       \
                         mean   min   max   mean   min   max      mean  min   
diagnosis                                                                     
Breast Cancer           11.85  11.2  12.5   6.40   5.0   7.8     220.0  190   
Colorectal Cancer       10.50  10.5  10.5   9.00   9.0   9.0     300.0  300   
Head & Neck Cancer      12.00  12.0  12.0   8.00   8.0   8.0     230.0  230   
Kidney Cancer           15.00  15.0  15.0   7.00   7.0   7.0     350.0  350   
Leukemia                 6.50   6.5   6.5   2.00   2.0   2.0      50.0   50   
Lung Cancer              9.55   9.1  10.0   5.35   4.5   6.2     170.0  160   
Lymphoma                 8.50   8.5   8.5   3.80   3.8   3.8     120.0  120   
Melanoma                13.00  13.0  13.0   6.00   6.0   6.0     220.0  220   
Multiple Myeloma         8.00   8.0   8.0   3.00   3.0   3.0     100.0  100   
Ovarian Cancer     

---
## Section 7: Build CSV

Export three CSV files:
1. `all_patients.csv` -- all 15+ patients
2. `high_risk_patients.csv` -- only high-risk patients
3. `drug_interaction_alerts.csv` -- drug interaction findings

In [ ]:
# === SECTION 7: BUILD CSV ===

# TODO: Export CSV 1 - All patients
df.to_csv(ALL_PATIENTS_FILE, index=False)
print(f"Exported all patients to {ALL_PATIENTS_FILE}")

# TODO: Export CSV 2 - High-risk patients only
high_risk_df = df[df['is_high_risk'] == True]
high_risk_df.to_csv(HIGH_RISK_PATIENTS_FILE, index=False)
print(f"Exported high-risk patients to {HIGH_RISK_PATIENTS_FILE}")

# TODO: Export CSV 3 - Drug interaction alerts
# Check if drug_interactions_df is empty before exporting to avoid creating an empty file with only headers
if not drug_interactions_df.empty:
    drug_interactions_df.to_csv(DRUG_INTERACTION_ALERTS_FILE, index=False)
    print(f"Exported drug interaction alerts to {DRUG_INTERACTION_ALERTS_FILE}")
else:
    print("No drug interaction alerts found, skipping export of drug_interaction_alerts.csv")


Exported all patients to all_patients.csv
Exported high-risk patients to high_risk_patients.csv
No drug interaction alerts found, skipping export of drug_interaction_alerts.csv


---
## Section 8: Email

In [ ]:
# === SECTION 8: EMAIL ===

# TODO: Print a simulated email summary showing:
# - Total patients processed
# - Number of high-risk patients
# - Number of drug interaction alerts
# - Files generated

total_patients = len(df)
num_high_risk = len(high_risk_df)
num_drug_alerts = len(drug_interactions_df)

email_summary = f"""
Subject: Daily Clinical Intake Summary - {pd.Timestamp.now().strftime('%Y-%m-%d')}

Dear Oncologist,

This is an automated summary of today's clinical patient intake.

Total patients processed: {total_patients}
Number of high-risk patients identified: {num_high_risk}
Number of drug interaction alerts: {num_drug_alerts}

Key alerts:
  - High-risk patients require immediate review. Please refer to '{HIGH_RISK_PATIENTS_FILE}'.
  - No drug interactions were found today. If any were, they would be in '{DRUG_INTERACTION_ALERTS_FILE}'.

For a complete list of all patients, please see '{ALL_PATIENTS_FILE}'.

Sincerely,
KHCC Automated System
"""

print(email_summary)



Subject: Daily Clinical Intake Summary - 2026-04-17

Dear Oncologist,

This is an automated summary of today's clinical patient intake.

Total patients processed: 15
Number of high-risk patients identified: 5
Number of drug interaction alerts: 0

Key alerts:
  - High-risk patients require immediate review. Please refer to 'high_risk_patients.csv'.
  - No drug interactions were found today. If any were, they would be in 'drug_interaction_alerts.csv'.

For a complete list of all patients, please see 'all_patients.csv'.

Sincerely,
KHCC Automated System



---
## Section 9: Markdown Summary

### Part 2 Summary
- **Patients processed:** [TODO]
- **High-risk patients:** [TODO]
- **Drug interactions found:** [TODO]
- **CSV files generated:** [TODO]
- **Key findings:** [TODO]

## Section 9: Markdown Summary

### Part 2 Summary
- **Patients processed:** 15
- **High-risk patients:** 5
- **Drug interactions found:** 0
- **CSV files generated:** `all_patients.csv`, `high_risk_patients.csv`

### Implementation Analysis (200-400 words)
This notebook successfully implements a foundational clinical intake automation pipeline. The core of the system is the `PatientIntake` class, designed to encapsulate patient-specific data such as demographics, lab results, medications, and clinical notes. Key methods within this class, `get_lab_alerts()`, `is_high_risk()`, and `to_dict()`, demonstrate robust logic for clinical assessment and data preparation.

The `get_lab_alerts()` method is crucial for flagging individual lab abnormalities against predefined clinical thresholds (e.g., `HGB_CRITICAL`, `WBC_HIGH`, `PLATELETS_LOW`). This allows for immediate identification of concerning lab values. Building upon this, `is_high_risk()` integrates multiple criteria to identify high-risk patients. For instance, a patient is flagged if they have critically low hemoglobin, two or more abnormal labs, or a combination of low WBC and specific chemotherapy agents like Cisplatin—logic directly reflecting oncology-specific concerns.

For medication safety, a separate `interactions` DataFrame was constructed, containing a list of known drug-drug interactions with associated severities and descriptions. The implementation then systematically checks each patient's medication list against this table (`elKGLqMu6Aza`). In this specific run with the provided sample data, no direct drug interactions were identified, which is a positive outcome for the cohort but highlights the importance of the mechanism for future data with diverse medication regimens.

The system culminates in structured data export and reporting. All processed patient data, enriched with risk assessments and alert counts, is consolidated into a pandas DataFrame (`df`) using the `to_dict()` method. This DataFrame is then used to generate `all_patients.csv` and a filtered `high_risk_patients.csv`, enabling quick access for oncologists to prioritize patient review. The absence of drug interaction alerts meant `drug_interaction_alerts.csv` was not created, demonstrating thoughtful handling of empty datasets.

Further analysis included grouping patient data by diagnosis to calculate summary statistics for lab values. This `groupby('diagnosis')` operation provides an overview of lab trends across different cancer types, which could be instrumental in developing diagnosis-specific monitoring protocols or identifying common complications. The email report generated as part of the stretch challenge further streamlines communication, providing a concise summary directly to clinicians.

---
## Part 4 -- Stretch Challenge (20%)

Choose ONE of the following options:

### Option A: Email Report
Write a function that generates a formatted email body summarizing the daily intake.

### Option B: Multi-Day Trend
Create intake data for 5 consecutive days (75+ patients). Analyze trends.

### Option C: Deployment Proposal
Write a 1-page markdown document proposing how this notebook could be deployed at KHCC.

In [ ]:
# === PART 4: STRETCH CHALLENGE ===

# Option A: Email Report
import pandas as pd

def generate_daily_intake_email_body(total_patients, num_high_risk, num_drug_alerts,
                                     high_risk_file, drug_alert_file, all_patients_file):
    """Generates a formatted email body summarizing the daily patient intake."""
    report_date = pd.Timestamp.now().strftime('%Y-%m-%d')

    email_body = f"""
Subject: Daily Clinical Intake Summary - {report_date}

Dear Oncologist,

This is an automated summary of today's clinical patient intake.

Total patients processed: {total_patients}
Number of high-risk patients identified: {num_high_risk}
Number of drug interaction alerts: {num_drug_alerts}

Key alerts:
  - High-risk patients require immediate review. Please refer to '{high_risk_file}'.
  - {'No drug interactions were found today.' if num_drug_alerts == 0 else f'Drug interactions were found. Please refer to \'{drug_alert_file}\' for details.'}

For a complete list of all patients, please see '{all_patients_file}'.

Sincerely,
KHCC Automated System
"""
    return email_body

# Demonstrate the function using the previously calculated values
email_output = generate_daily_intake_email_body(
    total_patients=total_patients,
    num_high_risk=num_high_risk,
    num_drug_alerts=num_drug_alerts,
    high_risk_file=HIGH_RISK_PATIENTS_FILE,
    drug_alert_file=DRUG_INTERACTION_ALERTS_FILE,
    all_patients_file=ALL_PATIENTS_FILE
)

print("\n--- Stretch Challenge: Option A - Email Report ---")
print(email_output)



--- Stretch Challenge: Option A - Email Report ---

Subject: Daily Clinical Intake Summary - 2026-04-17

Dear Oncologist,

This is an automated summary of today's clinical patient intake.

Total patients processed: 15
Number of high-risk patients identified: 5
Number of drug interaction alerts: 0

Key alerts:
  - High-risk patients require immediate review. Please refer to 'high_risk_patients.csv'.
  - No drug interactions were found today.

For a complete list of all patients, please see 'all_patients.csv'.

Sincerely,
KHCC Automated System



---
### Submission Checklist

- [ ] PatientIntake class with all 4 methods working
- [ ] 15+ unique patient records with varied data
- [ ] Drug interaction table with 10+ pairs
- [ ] GroupBy analysis by diagnosis
- [ ] 3 CSV files generated (all_patients, high_risk, drug_interactions)
- [ ] Part 3 analysis (200-400 words) referencing your implementation
- [ ] Part 4 stretch challenge completed
- [ ] Pushed to GitHub with 3+ commits
- [ ] No real patient data in the notebook